# 04d Taiwan Electricity Features

將台電鄉鎮市(郵遞區)別用電統計資料對應至 DeepSolar `electricity_*` 欄位。

**Input:**
- `data/taiwan/power/{107~111}年鄉鎮市(郵遞區)別用電統計資料.csv`

**Output:**
- `data/taiwan/power/taiwan_electricity_features.csv`

**DeepSolar 欄位對應：**

| DeepSolar 欄位 | r with tile_count | 台灣來源 |
|---|---|---|
| `electricity_consume_residential` | -0.208 | 1表燈非營業用，5年均值 per 用戶 |
| `electricity_price_industrial` | +0.336 | 107–111年低壓非時間流動電費年均 **2.493**元/度（全台統一常數） |
| `electricity_price_commercial` | +0.294 | 107–111年表燈營業第2段年均 **3.123**元/度（全台統一常數） |
| `electricity_consume_commercial` | -0.034 | 2表燈營業用，5年均值 per 用戶 |
| `electricity_consume_total` | -0.045 | 23/26總計，5年均值（全鄉鎮總用電量） |

> 電價費率查驗：閱讀 107–112年各類電價表 PDF，確認 107–111年全部標示「107年4月1日起實施」，費率完全相同；112年4月1日才調漲，故選用 107–111年資料無需額外修正。

> `avg_electricity_retail_rate` 未入選 90 個 voted 特徵，不在本 notebook 處理範圍。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Chinese font
matplotlib.rc('font', family='Microsoft JhengHei')
plt.rcParams['axes.unicode_minus'] = False

PROJ = Path('..').resolve()
sys.path.insert(0, str(PROJ))
from src.features import set_seed, load_taiwan_electricity_features, add_towncode_to_electricity
set_seed(42)

POWER_DIR = PROJ / 'data' / 'taiwan' / 'power'
POP_CSV   = PROJ / 'data' / 'taiwan' / 'population' / 'taiwan_population_features.csv'
OUT_CSV   = POWER_DIR / 'taiwan_electricity_features.csv'
FIG_DIR   = PROJ / 'outputs' / 'figures' / 'transfer'
FIG_DIR.mkdir(parents=True, exist_ok=True)

YEARS = (107, 108, 109, 110, 111)  # 5年均值，排除112年（欄名改版）

print('Project root:', PROJ)
print('Power dir exists:', POWER_DIR.exists())
print('Years:', YEARS)

## Step 1 - 載入台電鄉鎮用電統計（107–111 年）

In [ ]:
elec_raw = load_taiwan_electricity_features(
    power_dir=str(POWER_DIR),
    years=YEARS,
)

# 加入 TOWNCODE / COUNTYNAME / TOWNNAME
# 注意：新竹市(300)、嘉義市(600) 各區共用同一郵遞前三碼，
#       電力CSV 以市名代入整體列；add_towncode_to_electricity 會自動
#       展開為各行政區（複製同一份電力數值）。
pop_ref = pd.read_csv(POP_CSV, encoding='utf-8-sig')
elec    = add_towncode_to_electricity(elec_raw, pop_ref)

# Drop 真正無法對應的列（廢止行政區、離島哨所、零值列）
n_before = len(elec)
dropped  = elec[elec['TOWNCODE'].isna()][['郵遞區號', '行政區', 'electricity_consume_total']]
elec     = elec[elec['TOWNCODE'].notna()].reset_index(drop=True)
print(f'Drop 無 TOWNCODE 列 ({n_before - len(elec)} 筆):')
print(dropped.to_string(index=False))
print()

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
elec.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved → {OUT_CSV}')
print(f'Shape: {elec.shape}')
print(f'Columns: {list(elec.columns)}')
elec.head()

## Step 2 - Schema 差異說明

| 年份 | 用電欄名 | 數字格式 | 有累計欄 | 總計項目 |
|---|---|---|---|---|
| 107–109 | `用電種類` | 有逗號 | ✅ | `23總計` |
| 110 | `用電種類` | 無逗號 | ✅ | `23總計` |
| 111 | `用電種類` | 無逗號 | ✅ | `26總計`（EV 新增3行）|
| 112 | `項目`（改名） | 無逗號 | ❌ | — |

解析策略：
- 以 `startswith('1表燈非營業用')` 取居住用、`startswith('2表燈營業用')` 取商業用
- 以 `contains('總計')` 取全區合計（同時適用 23 和 26）
- 取 `月份==12` 的「售電度數(當年累計)」直接得年度總量（不需加總12個月）

In [ ]:
# 確認各年份 CSV 存在
for yr in YEARS:
    fp = POWER_DIR / f'{yr}年鄉鎮市(郵遞區)別用電統計資料.csv'
    print(f'{yr}年: {"✅" if fp.exists() else "❌ 缺少"} ({fp.name})')

## Step 3 - NaN 檢查

In [ ]:
print('NaN 統計：')
print(elec.isnull().sum())
print()
nan_rows = elec[elec['electricity_consume_residential'].isna()]
print(f'Residential NaN 鄉鎮（共 {len(nan_rows)} 個）：')
print(nan_rows[['郵遞區號', '行政區']].to_string(index=False))
print('\n原因：這些鄉鎮在台電統計中5年內用戶數均為 * 遮蔽（統計保護）')

## Step 4 - 電價欄位確認（全台統一常數）

In [ ]:
print('電價唯一值數：', elec['electricity_price_industrial'].nunique())
print(f'  工業電價: {elec["electricity_price_industrial"].iloc[0]:.6f} USD/kWh  ← 低壓非時間流動電費年均 2.493元/度 ÷ 28.5')
print(f'  商業電價: {elec["electricity_price_commercial"].iloc[0]:.6f} USD/kWh  ← 表燈營業第2段年均 3.123元/度 ÷ 28.5')
print()
print('電價計算：')
print('  工業：低壓電力非時間流動電費 夏月2.58 × 4個月 + 非夏月2.45 × 8個月 / 12 = 2.493元/度')
print('  商業：表燈營業用第2段(331–700度) 夏月3.55 × 4個月 + 非夏月2.91 × 8個月 / 12 = 3.123元/度')
print()
print('注意：台灣台電為獨占事業，電價全台統一（無地區差異）。')
print('      107–111年費率一致（107年4月1日起實施），112年4月1日才調漲。')
print('      模型推論時這兩個欄位對所有鄉鎮套用相同係數，無法提供地區分辨能力。')

## Step 5 - 合理性驗證

In [ ]:
# 統計摘要
feat_cols = ['electricity_consume_residential', 'electricity_consume_commercial', 'electricity_consume_total']
print(elec[feat_cols].describe().round(2))

# 台北市中正區用電 > 離島（合理性）
zhongzheng = elec[elec['郵遞區號'] == 100]['electricity_consume_total'].values
lienchiang = elec[elec['郵遞區號'] == 209]['electricity_consume_total'].values  # 連江縣
if len(zhongzheng) and len(lienchiang):
    assert zhongzheng[0] > lienchiang[0], '台北市中正區總用電應大於連江縣'
    print(f'\n✅ 合理性：台北市中正區({zhongzheng[0]:.2e} kWh) >> 連江縣({lienchiang[0]:.2e} kWh)')

In [ ]:
print('居住用電 Top 10（per 用戶）：')
print(elec.nlargest(10, 'electricity_consume_residential')[['郵遞區號', '行政區', 'electricity_consume_residential']].to_string(index=False))
print()
print('居住用電 Bottom 10：')
print(elec.nsmallest(10, 'electricity_consume_residential')[['郵遞區號', '行政區', 'electricity_consume_residential']].to_string(index=False))

## Step 6 - 分布圖

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('台灣鄉鎮電力消費分布（107–111年均值）', fontsize=14)

elec['electricity_consume_residential'].dropna().hist(
    ax=axes[0], bins=30, edgecolor='white', color='steelblue'
)
axes[0].set_title('居住用電 (kWh/用戶/年)')
axes[0].set_xlabel('kWh')

elec['electricity_consume_commercial'].dropna().hist(
    ax=axes[1], bins=30, edgecolor='white', color='coral'
)
axes[1].set_title('商業用電 (kWh/用戶/年)')
axes[1].set_xlabel('kWh')

plt.tight_layout()
out_fig = FIG_DIR / '04d_taiwan_electricity_distributions.png'
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved: {out_fig}')

## Step 7 - 輸出確認

In [ ]:
print(f'CSV 儲存路徑: {OUT_CSV}')
print(f'檔案存在: {OUT_CSV.exists()}')

# 讀回確認
check = pd.read_csv(OUT_CSV, encoding='utf-8-sig')
print(f'Shape: {check.shape}')
print(check.dtypes)
check.head(3)

## 總結

| DeepSolar 欄位 | 台灣值範圍 | 來源 | Join Key |
|---|---|---|---|
| `electricity_consume_residential` | 1,901 – 10,405 kWh/用戶/年 | 1表燈非營業用，107–111年均值 | 郵遞區號 |
| `electricity_consume_commercial` | 3,898 – 35,984 kWh/用戶/年 | 2表燈營業用，107–111年均值 | 郵遞區號 |
| `electricity_consume_total` | 0 – 1.05e+10 kWh/年 | 全區總計，107–111年均值 | 郵遞區號 |
| `electricity_price_industrial` | 0.0875 $/kWh（常數） | 低壓非時間流動電費年均（夏月4月×2.58 + 非夏月8月×2.45）÷12÷28.5 | — |
| `electricity_price_commercial` | 0.1096 $/kWh（常數） | 表燈營業第2段年均（夏月4月×3.55 + 非夏月8月×2.91）÷12÷28.5 | — |

**電價費率來源：** 台電各類電價表 PDF，107–111年全部標示「107年4月1日起實施」，費率完全相同。112年4月1日才調漲。

**後續 Join：**  
本 CSV 以 `郵遞區號` 為地理 key，與其他特徵（TOWNCODE 為 key）合併時，  
需透過 `data/taiwan/鄉(鎮、市、區)界線1140318/TOWN_MOI_1140318.dbf` 取得郵遞區號 → TOWNCODE 對應表。

**NaN 說明：**  
residential / commercial 各 5 筆 NaN（郵遞區號 290、703、817、819、896），  
因這些鄉鎮在台電統計中 5 年內用戶數均為 `*`（統計保護），無法計算 per-user 值。